# DML IRM vs CausalML NearestNeighborMatch

Causal Inference consists of two main parts: Identification Assumptions and Model Specification. SUTVA, Unconfoundedness, Overlap are strong assumptions that must be true to call our inference causal. Studies and quasi experiments often have problems with Identification Assumptions so in practice you spend time to prove them, not model specification

However, In this notebook I will focus on the model specification. Propensity Score matching is a classical ML non-parametric approcah, estimating ATTE. It must perform worse than DML approach because:

- uses both the propensity model and the outcome model, not just propensity scores

- is more robust to small model misspecification through orthogonalization

- uses cross-fitting, which reduces overfitting bias from ML nuisance models

- usually has lower bias than plain propensity score matching

- usually has lower variance / better efficiency because it keeps more information

- does not throw away as much data as matching often does

- works much better in high-dimensional and nonlinear settings

- provides more principled statistical inference (standard errors, confidence intervals)

- can estimate ATE or ATTE cleanly, not just the treated-group effect by default

We will compare absolute estimates on DGPs from Causalis between IRM DML model implemented in Causalis and NearestNeighborMatch implemented in CausalML

In [1]:
import time
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, CatBoostRegressor
from causalml.match import NearestNeighborMatch

from causalis.data_contracts import CausalData
from causalis.scenarios.unconfoundedness import IRM
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_26_rich
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_binary_26

# generate_obs_hte_26_rich()

Read more about dgp at https://causalis.causalcraft.com/articles/generate_obs_hte_26_rich

In [2]:
ORACLE_COLS = {"m", "m_obs", "tau_link", "g0", "g1", "cate"}
TREATMENT_COL = "d"
OUTCOME_COL = "y"
SIZES = [10_000, 100_000, 1_000_000]
SEED = 42


def infer_confounders(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in ORACLE_COLS.union({TREATMENT_COL, OUTCOME_COL})]


def estimate_irm_atte(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float]:
    t0 = time.perf_counter()
    cd = CausalData(
        df=df[[OUTCOME_COL, TREATMENT_COL] + confounders].copy(),
        treatment=TREATMENT_COL,
        outcome=OUTCOME_COL,
        confounders=confounders,
    )

    irm = IRM(
        random_state=seed,
        ml_g=CatBoostRegressor(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
        ml_m=CatBoostClassifier(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
    )

    atte = float(irm.fit(cd).estimate(score="ATTE", diagnostic_data=False).value)
    return atte, (time.perf_counter() - t0)


def estimate_matching_atte_total(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float, int]:
    t0 = time.perf_counter()

    ps_model = CatBoostClassifier(
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
        random_seed=seed,
    )
    ps_model.fit(df[confounders], df[TREATMENT_COL].astype(int), verbose=False)

    work = df.copy()
    work["propensity_score"] = np.clip(
        ps_model.predict_proba(work[confounders])[:, 1],
        1e-2,
        1 - 1e-2,
    )

    psm = NearestNeighborMatch(replace=True, ratio=1, random_state=seed)
    matched = psm.match(
        data=work,
        treatment_col=TREATMENT_COL,
        score_cols=["propensity_score"],
    )

    atte = float(
        matched.loc[matched[TREATMENT_COL] == 1, OUTCOME_COL].mean()
        - matched.loc[matched[TREATMENT_COL] == 0, OUTCOME_COL].mean()
    )

    return atte, (time.perf_counter() - t0), int(matched.shape[0])


In [3]:
results = []

for n in SIZES:
    print(f"Running n={n:,} ...")

    df = generate_obs_hte_26_rich(
        n=n,
        seed=SEED,
        include_oracle=True,
        return_causal_data=False,
    )
    confounders = infer_confounders(df)

    ground_truth_atte = float(df.loc[df[TREATMENT_COL] == 1, "cate"].mean())
    irm_atte, irm_runtime_sec = estimate_irm_atte(df=df, confounders=confounders, seed=SEED)

    matching_atte, matching_runtime_sec, matched_n = estimate_matching_atte_total(
        df=df,
        confounders=confounders,
        seed=SEED,
    )

    results.append(
        {
            "n": n,
            "ground_truth_atte": ground_truth_atte,
            "irm_atte": irm_atte,
            "matching_atte": matching_atte,
            "irm_abs_error": abs(irm_atte - ground_truth_atte),
            "matching_abs_error": abs(matching_atte - ground_truth_atte),
            "irm_runtime_sec": irm_runtime_sec,
            "matching_runtime_sec": matching_runtime_sec,
            "matched_n": matched_n,
        }
    )

comparison = pd.DataFrame(results)
comparison

Running n=10,000 ...
Running n=100,000 ...
Running n=1,000,000 ...


,n,ground_truth_atte,irm_atte,matching_atte,irm_abs_error,matching_abs_error,irm_runtime_sec,matching_runtime_sec,matched_n
0,10000,11.454404,6.256087,35.978189,5.198317,24.523785,9.647452,1.357374,618
1,100000,10.914991,12.106856,28.721069,1.191864,17.806077,27.374003,3.540408,9238
2,1000000,11.028129,10.340542,16.589285,0.687587,5.561156,192.813909,29.992732,99218


DML IRM outperforms NearestNeighborMatch

# generate_obs_hte_binary_26()

In [4]:
for _, row in comparison.iterrows():
    n = int(row["n"])
    print(
        f"n={n:,}: ground truth ATTE={row['ground_truth_atte']:.6f}, "
        f"IRM ATTE={row['irm_atte']:.6f}, matching ATTE={row['matching_atte']:.6f}"
    )

n=10,000: ground truth ATTE=11.454404, IRM ATTE=6.256087, matching ATTE=35.978189
n=100,000: ground truth ATTE=10.914991, IRM ATTE=12.106856, matching ATTE=28.721069
n=1,000,000: ground truth ATTE=11.028129, IRM ATTE=10.340542, matching ATTE=16.589285


read more about the dgp at https://causalis.causalcraft.com/articles/generate_obs_hte_binary_26

In [5]:
ORACLE_COLS = {"m", "m_obs", "tau_link", "g0", "g1", "cate"}
TREATMENT_COL = "d"
OUTCOME_COL = "y"
SIZES = [10_000, 100_000, 1_000_000]
SEED = 42


def infer_confounders(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in ORACLE_COLS.union({TREATMENT_COL, OUTCOME_COL})]


def estimate_irm_atte(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float]:
    t0 = time.perf_counter()
    cd = CausalData(
        df=df[[OUTCOME_COL, TREATMENT_COL] + confounders].copy(),
        treatment=TREATMENT_COL,
        outcome=OUTCOME_COL,
        confounders=confounders,
    )

    irm = IRM(
        random_state=seed,
        ml_g=CatBoostRegressor(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
        ml_m=CatBoostClassifier(
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
            random_seed=seed,
        ),
    )

    atte = float(irm.fit(cd).estimate(score="ATTE", diagnostic_data=False).value)
    return atte, (time.perf_counter() - t0)


def estimate_matching_atte_total(df: pd.DataFrame, confounders: list[str], seed: int) -> tuple[float, float, int]:
    t0 = time.perf_counter()

    ps_model = CatBoostClassifier(
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
        random_seed=seed,
    )
    ps_model.fit(df[confounders], df[TREATMENT_COL].astype(int), verbose=False)

    work = df.copy()
    work["propensity_score"] = np.clip(
        ps_model.predict_proba(work[confounders])[:, 1],
        1e-2,
        1 - 1e-2,
    )

    psm = NearestNeighborMatch(replace=True, ratio=1, random_state=seed)
    matched = psm.match(
        data=work,
        treatment_col=TREATMENT_COL,
        score_cols=["propensity_score"],
    )

    atte = float(
        matched.loc[matched[TREATMENT_COL] == 1, OUTCOME_COL].mean()
        - matched.loc[matched[TREATMENT_COL] == 0, OUTCOME_COL].mean()
    )

    return atte, (time.perf_counter() - t0), int(matched.shape[0])


In [6]:
results = []

for n in SIZES:
    print(f"Running n={n:,} ...")

    df = generate_obs_hte_binary_26(
        n=n,
        seed=SEED,
        include_oracle=True,
        return_causal_data=False,
    )
    confounders = infer_confounders(df)

    ground_truth_atte = float(df.loc[df[TREATMENT_COL] == 1, "cate"].mean())
    irm_atte, irm_runtime_sec = estimate_irm_atte(df=df, confounders=confounders, seed=SEED)

    matching_atte, matching_runtime_sec, matched_n = estimate_matching_atte_total(
        df=df,
        confounders=confounders,
        seed=SEED,
    )

    results.append(
        {
            "n": n,
            "ground_truth_atte": ground_truth_atte,
            "irm_atte": irm_atte,
            "matching_atte": matching_atte,
            "irm_abs_error": abs(irm_atte - ground_truth_atte),
            "matching_abs_error": abs(matching_atte - ground_truth_atte),
            "irm_runtime_sec": irm_runtime_sec,
            "matching_runtime_sec": matching_runtime_sec,
            "matched_n": matched_n,
        }
    )

comparison = pd.DataFrame(results)
comparison

Running n=10,000 ...
Running n=100,000 ...
Running n=1,000,000 ...


,n,ground_truth_atte,irm_atte,matching_atte,irm_abs_error,matching_abs_error,irm_runtime_sec,matching_runtime_sec,matched_n
0,10000,0.103885,0.102344,0.115652,0.001541,0.011767,11.177699,1.624111,2594
1,100000,0.101238,0.103547,0.074051,0.002309,0.027187,33.497727,4.216389,29304
2,1000000,0.101411,0.103282,0.094218,0.001871,0.007192,219.734037,31.273997,298710


In [7]:
for _, row in comparison.iterrows():
    n = int(row["n"])
    print(
        f"n={n:,}: ground truth ATTE={row['ground_truth_atte']:.6f}, "
        f"IRM ATTE={row['irm_atte']:.6f}, matching ATTE={row['matching_atte']:.6f}"
    )

n=10,000: ground truth ATTE=0.103885, IRM ATTE=0.102344, matching ATTE=0.115652
n=100,000: ground truth ATTE=0.101238, IRM ATTE=0.103547, matching ATTE=0.074051
n=1,000,000: ground truth ATTE=0.101411, IRM ATTE=0.103282, matching ATTE=0.094218


DML IRM outperforms NearestNeighborMatch

# Conclusion

I recommend to use DML IRM for Unconfoundedness scenario as default model specification